In [1]:
!pip install -q sentence-transformers faiss-cpu transformers datasets pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

In [5]:
import pandas as pd

# Convert the loaded dataset to a pandas DataFrame
df = dataset['train'].to_pandas()

In [7]:
df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [8]:
print(df.shape)

(117592, 4)


In [9]:
print(df.columns)

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='object')


In [10]:
df["abstract"]

,abstract
0,The problem of statistical learning is to co...
1,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem is conside...
3,Ordinal regression is an important type of l...
4,This paper uncovers and explores the close r...
...,...
117587,The sharing of fake news and conspiracy theori...
117588,The feature subset selection problem aims at s...
117589,We present a short and elementary proof of the...
117590,"We introduce CoLN, Combined Learning of Neural..."


In [11]:
print(df.iloc[0]["title"])
print()
print(df.iloc[0]["abstract"])

Learning from compressed observations

  The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regres

In [12]:
df.head()
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117592 entries, 0 to 117591
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Unnamed: 0.1  117592 non-null  int64  
 1   Unnamed: 0    112592 non-null  float64
 2   title         117592 non-null  object 
 3   abstract      117592 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 3.6+ MB


### Combining Title and Abstract for 'paper_text'

Combining the 'title' and 'abstract' into a single `paper_text` column is a crucial design decision for improving semantic search accuracy. Here's why:

1.  **Rich Context**: The title provides a concise summary, while the abstract offers a more detailed overview of the paper's content, methodology, and results. Merging them creates a richer, more comprehensive text representation.

2.  **Improved Embeddings**: When using techniques like sentence embeddings (which we will use later), a combined text often leads to more robust and semantically meaningful vector representations. The additional context from the abstract helps the embedding model capture nuances that might be missed if only the title were used.

3.  **Enhanced Search Relevance**: For semantic search, the goal is to find documents that are conceptually similar to a query. By feeding a more complete `paper_text` into the embedding model, the resulting vectors will better represent the paper's core ideas, leading to more accurate and relevant search results compared to relying solely on the title or abstract.

4.  **Reduced Ambiguity**: Titles can sometimes be vague or overly broad. The abstract clarifies the specific scope and contribution of the work, reducing ambiguity and ensuring that the generated embeddings are precise.

In [ ]:
df['paper_text'] = df['title'] + '. ' + df['abstract']
display(df[['title', 'abstract', 'paper_text']].head())

In [16]:
df['paper_text'] = df['title'] + '. ' + df['abstract']
print(df["paper_text"].iloc[0])

Learning from compressed observations.   The problem of statistical learning is to construct a predictor of a random
variable $Y$ as a function of a related random variable $X$ on the basis of an
i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable
predictors are drawn from some specified class, and the goal is to approach
asymptotically the performance (expected loss) of the best predictor in the
class. We consider the setting in which one has perfect observation of the
$X$-part of the sample, while the $Y$-part has to be communicated at some
finite bit rate. The encoding of the $Y$-values is allowed to depend on the
$X$-values. Under suitable regularity conditions on the admissible predictors,
the underlying family of probability distributions and the loss function, we
give an information-theoretic characterization of achievable predictor
performance in terms of conditional distortion-rate functions. The ideas are
illustrated on the example of nonparametric regres

In [15]:
df['paper_text'] = df['title'] + '. ' + df['abstract']
empty_rows = (df["paper_text"].str.strip() == "").sum()
print("Empty papers:", empty_rows)

Empty papers: 0


In [19]:
from sentence_transformers import SentenceTransformer

In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
print(model)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [21]:
sample_text = df["paper_text"].iloc[0]

embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [22]:
print(embedding[:10])

[-0.13028255 -0.00723163 -0.00406495  0.0325141   0.11117471  0.01139132
  0.10118811 -0.08725087  0.04506742 -0.01332956]


In [23]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
sample_text = df["paper_text"].iloc[0]
embedding = model.encode(sample_text)

print(type(embedding))
print(embedding.shape)
print(embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<class 'numpy.ndarray'>
(384,)
[-0.13028255 -0.00723163 -0.00406495  0.0325141   0.11117471  0.01139132
  0.10118811 -0.08725087  0.04506742 -0.01332956]


In [25]:
papers = df["paper_text"].tolist()

In [ ]:
embeddings = model.encode(
    papers,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/3675 [00:00<?, ?it/s]

In [ ]:
print(type(embeddings))
print(embeddings.shape)
print(embeddings[0])

import numpy as np
np.save("paper_embeddings.npy", embeddings)

In [ ]:
embeddings = np.load("paper_embeddings.npy")

In [ ]:
import faiss
dimension = embeddings.shape[1]
print(dimension)
dimension = 512
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(index.ntotal)
faiss.write_index(index, "paper_faiss.index")

In [ ]:
index = faiss.read_index("paper_faiss.index")

In [ ]:
search_paper("Vision Transformer for Medical Imaging")

In [ ]:
def search_papers(query, top_k=5):
    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Search the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    return scores, indices

In [ ]:
scores, indices = search_papers(
    "Vision Transformer"
)

print(scores)
print(indices)

In [ ]:
for idx in indices[0]:
    print(df.iloc[idx]["title"])

In [ ]:
for idx in indices[0]:

    print("=" * 100)

    print("TITLE:")
    print(df.iloc[idx]["title"])

    print()

    print("ABSTRACT:")
    print(df.iloc[idx]["abstract"])

    print()

In [ ]:
for score, idx in zip(scores[0], indices[0]):

    print("=" * 100)

    print(f"Similarity Score : {score:.4f}")

    print(df.iloc[idx]["title"])

    print()

    print(df.iloc[idx]["abstract"])

In [ ]:
from transformers import pipeline

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

In [ ]:
abstract = df.iloc[0]["abstract"]
print(abstract)

In [ ]:
summary = summarizer(
    abstract,
    max_length=120,
    min_length=40,
    do_sample=False
)
print(summary[0]["summary_text"])

In [ ]:
def summarize_paper(text):
    summary = summarizer(
        text,
        max_length=120,
        min_length=40,
        do_sample=False
    )
    return summary[0]["summary_text"]

In [ ]:
scores, indices = search_papers(
    "Vision Transformer"
)

In [ ]:
for score, idx in zip(scores[0], indices[0]):
    print("="*80)
    print("Title:")
    print(df.iloc[idx]["title"])
    print()
    print("Summary:")
    summary = summarize_paper(
        df.iloc[idx]["abstract"]
    )
    print(summary)

In [ ]:
scores, indices = search_papers(
    query,
    top_k=5
)

def build_context(indices):

    context = ""

    for idx in indices[0]:

        title = df.iloc[idx]["title"]
        abstract = df.iloc[idx]["abstract"]

        context += f"""
Title:
{title}

Abstract:
{abstract}

-----------------------------------
"""

    return context

In [ ]:
context = build_context(indices)
print(context[:1000])

In [ ]:
prompt = f"""
You are an AI Research Assistant.

Use ONLY the information below.

If the answer is not present,
say that the papers do not contain
sufficient information.

Research Papers

{context}

Question:

{query}

Provide

1. Answer

2. Key Findings

3. Mention which papers contributed.
"""

In [ ]:
response = model.generate_content(prompt)

In [ ]:
context += f"""
Paper {i+1}

Title:
{title}

Abstract:
{abstract}
"""

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

st.title(" AI Research Assistant")
st.write(
    "Search Machine Learning research papers using Semantic Search and AI."
)
query = st.text_input(
    "Enter Research Topic"
)
search = st.button("Search")

index = faiss.read_index(
    "paper_faiss.index"
)
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

if search:

    scores, indices = search_papers(
        query
    )
for score, idx in zip(scores[0], indices[0]):

    st.subheader(df.iloc[idx]["title"])

    st.write(
        f"Similarity : {score:.4f}"
    )

    st.write(
        summarize_paper(
            df.iloc[idx]["abstract"]
        )
    )
with st.expander(
    "Read Full Abstract"
):
    st.write(
        df.iloc[idx]["abstract"]
    )


In [ ]:
page = st.sidebar.selectbox(
    "Menu",
    [
        "Search",
        "Analytics",
        "About"
    ]
)

In [ ]:
test_queries = [
    "Vision Transformer",
    "Large Language Models",
    "Reinforcement Learning",
    "Diffusion Models",
    "Graph Neural Networks",
    "Medical Image Segmentation",
    "Federated Learning",
    "Speech Recognition",
    "Object Detection",
    "Recommendation Systems"
]

In [ ]:
def precision_at_k(retrieved, relevant, k):

    retrieved = retrieved[:k]

    hits = len(set(retrieved) & set(relevant))

    return hits / k

In [ ]:
def recall_at_k(retrieved, relevant, k):

    retrieved = retrieved[:k]

    hits = len(set(retrieved) & set(relevant))

    return hits / len(relevant)

In [ ]:
import time

start = time.time()

scores, indices = search_papers(
    "Vision Transformer"
)

end = time.time()

print(end-start)

In [ ]:
import matplotlib.pyplot as plt

models = ["MiniLM","BGE","E5"]

precision = [0.82,0.87,0.89]

plt.bar(models, precision)

plt.title("Precision Comparison")

plt.show()